# Treino do modelo — Precificação em Copacabana

Executa o pipeline completo no Google Colab, que já vem com PyTorch instalado.
O projeto é clonado direto do GitHub, então este notebook testa exatamente o
que está publicado no repositório.

**Você precisa ter em mãos:** `listings.csv.gz` e, opcionalmente,
`calendar.csv.gz`, baixados em https://insideairbnb.com/get-the-data (seção Rio
de Janeiro). Eles não estão no repositório por causa do tamanho.

Execute as células em ordem com `Shift + Enter`.

## 1. Conferir o ambiente

In [ ]:
import sys
import torch
import numpy as np
import pandas as pd

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"pandas  : {pd.__version__}")
print(f"Dispositivo: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Clonar o repositório

In [ ]:
import os
import shutil

if os.path.exists('precificacao-copacabana'):
    shutil.rmtree('precificacao-copacabana')

!git clone -q https://github.com/jpmf-dev/precificacao-copacabana.git
%cd precificacao-copacabana

!git log --oneline | head -5
print()
!ls

## 3. Enviar os dados

Ao executar, aparece um botão para escolher arquivos. Selecione o
`listings.csv.gz` (e o `calendar.csv.gz`, se quiser o índice sazonal).

O upload leva alguns minutos: 23 MB e 43 MB.

In [ ]:
from google.colab import files

enviados = files.upload()

for nome in enviados:
    destino = f"data/raw/{nome}"
    shutil.move(nome, destino)
    print(f"  {nome} -> {destino} ({os.path.getsize(destino)/1e6:.1f} MB)")

!ls -la data/raw/

## 4. Suíte de testes

Confirma que o que está publicado funciona aqui. Deve terminar em `OK`, sem
nenhum `skipped`.

In [ ]:
!python -m unittest discover -s tests 2>&1 | tail -5

## 5. Treinar o modelo

Pipeline completo: carrega, limpa, calcula os baselines, treina, avalia e salva.

**Guarde esta saída** — os números vão para os slides 11 e 13.

In [ ]:
!python main.py treinar

## 6. Curva de convergência

In [ ]:
sys.path.insert(0, os.getcwd())

from src import config
from src.data import loader
from src.preprocessing import cleaning, features
from src.training import trainer
from src.evaluation import metrics, baselines
import matplotlib.pyplot as plt

limpo = cleaning.preparar_anuncios(loader.filtrar_bairro(loader.carregar_listings()))
X, y, nomes = features.montar_matriz(limpo)
xt, xv, yt, yv = features.dividir_treino_teste(X, y)
xtp, xvp, media, desvio = features.padronizar(xt, xv)

modelo, hist = trainer.treinar(xtp, yt, xvp, yv, verbose=False)
previsto = trainer.prever(modelo, xvp)

mae = metrics.erro_absoluto_medio(yv, previsto)
dentro = metrics.proporcao_dentro_tolerancia(yv, previsto)
print(f"MAE teste: R$ {mae:.2f} | dentro de +-25%: {dentro:.1%}")

plt.figure(figsize=(9, 5))
plt.plot(hist.epocas, hist.perda_treino, label='Treino', marker='o')
plt.plot(hist.epocas, hist.perda_teste, label='Teste', marker='s')
plt.xlabel('Epoca')
plt.ylabel('MAE (R$)')
plt.title('Convergencia do treinamento')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('curva_perda.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Comparação com os baselines

In [ ]:
prev_global = baselines.baseline_mediana_global(yt, yv)
mae_global = metrics.erro_absoluto_medio(yv, prev_global)
dentro_global = metrics.proporcao_dentro_tolerancia(yv, prev_global)

idx = features.dividir_treino_teste(np.arange(len(limpo)).reshape(-1, 1), y)
df_t = limpo.iloc[idx[0].ravel()]
df_v = limpo.iloc[idx[1].ravel()]
prev_grupo = baselines.baseline_mediana_por_grupo(df_t, df_v, ['accommodates', 'room_type'])
real = df_v[config.COLUNA_ALVO].to_numpy()
mae_grupo = metrics.erro_absoluto_medio(real, prev_grupo)
dentro_grupo = metrics.proporcao_dentro_tolerancia(real, prev_grupo)

rotulos = ['Mediana\nglobal', 'Mediana por\ncapacidade+tipo', 'Rede neural\nMLP']
maes = [mae_global, mae_grupo, mae]
acertos = [dentro_global * 100, dentro_grupo * 100, dentro * 100]
cores = ['#bbbbbb', '#888888', '#2a6ebb']

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
b1 = eixos[0].bar(rotulos, maes, color=cores)
eixos[0].set_ylabel('MAE (R$)')
eixos[0].set_title('Erro absoluto medio - menor e melhor')
eixos[0].bar_label(b1, fmt='R$ %.0f')

b2 = eixos[1].bar(rotulos, acertos, color=cores)
eixos[1].set_ylabel('% dentro de +-25%')
eixos[1].set_title('Estimativas utilizaveis - maior e melhor')
eixos[1].axhline(60, ls='--', c='red', label='Meta RNF02')
eixos[1].bar_label(b2, fmt='%.1f%%')
eixos[1].legend()

plt.tight_layout()
plt.savefig('comparacao_baselines.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"RNF01 - superar R$ {mae_grupo:.2f}: {'ATENDIDO' if mae < mae_grupo else 'NAO ATENDIDO'}")
print(f"RNF02 - ao menos 60% dentro de +-25%: {'ATENDIDO' if dentro >= 0.60 else 'NAO ATENDIDO'}")

## 8. Experimentos com diferentes configurações

Alimenta o slide 11, que pede configurações comparadas. Nove combinações: três
tamanhos de camada oculta por três taxas de aprendizado.

In [ ]:
resultados = []

for oculta in [16, 32, 64]:
    for lr in [0.005, 0.01, 0.05]:
        m, _ = trainer.treinar(
            xtp, yt, xvp, yv,
            taxa_aprendizado=lr,
            dim_oculta=oculta,
            verbose=False,
        )
        p = trainer.prever(m, xvp)
        resultados.append({
            'neuronios': oculta,
            'learning_rate': lr,
            'MAE': round(metrics.erro_absoluto_medio(yv, p), 2),
            'dentro_25pct': round(metrics.proporcao_dentro_tolerancia(yv, p) * 100, 1),
        })

tabela = pd.DataFrame(resultados).sort_values('MAE')
print(tabela.to_string(index=False))
tabela.to_csv('experimentos.csv', index=False)

melhor = tabela.iloc[0]
print()
print(f"Melhor: {melhor['neuronios']:.0f} neuronios, lr={melhor['learning_rate']}, MAE R$ {melhor['MAE']:.2f}")

## 9. Índice sazonal e estimativa

Só execute se enviou o `calendar.csv.gz`.

In [ ]:
!python main.py sazonal

In [ ]:
!python main.py estimar

## 10. Baixar os resultados

In [ ]:
files.download('curva_perda.png')
files.download('comparacao_baselines.png')
files.download('experimentos.csv')
files.download('models/modelo_precificacao.pth')

---

## Depois de rodar

1. **Copie a saída da célula 5** e a tabela da célula 8
2. **Baixe os gráficos** e commite em `docs/resultados/`
3. **Salve este notebook** (Arquivo → Fazer download → .ipynb) e commite em
   `notebooks/treino.ipynb`